| Field | Details |
|:---|:---|
| **Project** | ML Research Paper Summarizer |
| **Author** | Madeti Karthikeya |
| **Date** | June 2026 |
| **Libraries used** | Selenium, OpenAI, tiktoken, pypdf |

---

## Project Overview:
**Intent:** To create a research paper summarizer for myself which would help me to understand an overall perspective of a Machine Learning research paper which I can extract from Papers with Code platform. This would an enabler for me to stay updated with the latest trends and researchers, helping me identify knowledge gaps and new aspects of learning which I can implement in my work.

## Project Flow:
1. Extracting PDF URL
2. Input token estimations
3. Cleaning and extracting text from PDF
4. Creating prompts and Setting up Model parameters
5. LLM Calling
6. Comparison of results between GPT4 Mini and Locally run Phi3 model
7. Exploring Challenges Furthur


In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from selenium import webdriver
from selenium.webdriver.common.by import By


from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


import time
import requests
from io import BytesIO
from pypdf import PdfReader
import re
import tiktoken

In [ ]:
# Opening a browser:
def extract_url(paper_name):
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 10)

    driver.get("https://paperswithcode.co/")

    search = driver.find_element(By.CLASS_NAME,"search-trigger")
    search = wait.until(
    EC.element_to_be_clickable(
        (By.CLASS_NAME, "search-trigger"))
    )
    search.click()

    # modal = driver.find_element(By.CLASS_NAME,"search-modal")
    modal = wait.until(
    EC.visibility_of_element_located(
        (By.CLASS_NAME, "search-modal")
    ))

    search_box = modal.find_element(By.CLASS_NAME,"search-modal-input")
    search_box.send_keys(paper_name,Keys.ENTER) # EXTACT PAPER NAME

    # result = driver.find_element(By.CLASS_NAME,"search-result")
    result = wait.until(
    EC.element_to_be_clickable(
        (By.CLASS_NAME, "search-result")
    ))

    result.click()

    # wait.until(
    # lambda d: "paperswithcode.com/paper/" in d.current_url)

    time.sleep(3)

    # pdf = driver.find_element(By.CLASS_NAME,"action-btn")
    pdf = wait.until(
    EC.presence_of_element_located(
        (By.CLASS_NAME, "action-btn")
    ))
    
    pdf_url = pdf.get_attribute("href")
    
    driver.quit()

    return pdf_url

In [ ]:
print(extract_url("Adaptation of Agentic AI"))

In [ ]:
# Analysing Input tokens 
print('Number of characters',len(text),'and word',len(text.split()))


encoding = tiktoken.encoding_for_model("gpt-5")


num_tokens = len(encoding.encode(text))
print('with model gpt-5, number of input tokens',num_tokens)
print('for a research paper, tokens to words ratios: ',round(num_tokens/len(text.split()),2) )

### Building my own Technical / Research Tutor
MULTI LLM CALLS -> MULTI SHOT PROMPTING -> ANSWER VERSIONS WITH MULTIPLE MODELS
#### Intent: 
To Explain the research paper in a simple language for understanding why the paper exsists:
1. What the Author wants to demonstrate
2. What value the paper brings to the world
3. Understand basic concepts which you don't know before you look into the paper
4. Understand the key sections - going over the surface: Abstract, conclusion, data, results 
5. Entering the subspace: Model architecture -> Review Input and Outputs -> Highlight new techiniques -> Isolate loss function -> Review training parameters

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
def extract_pdf_text(pdf_url):
    pdf_bytes = requests.get(pdf_url).content
    reader = PdfReader(BytesIO(pdf_bytes))

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    
    patterns = [
        r"\nReferences\s*\n",
        r"\nREFERENCES\s*\n",
        r"\nBibliography\s*\n",
        r"\nBIBLIOGRAPHY\s*\n"
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            text = text[:match.start()]
            break

    return text

def llm_msg(pdf_url):
    text = extract_pdf_text(pdf_url)

    system_prompt = """Hi, you are an expert assistant who is helpful in creating and summarizing the machine learning research papers. Since you have already worked in companies like Google Deep Research, you are developers who are young, fresh out of college, to understand the machine learning concepts as well as the new concepts which are evolving in research papers. This is so that they can use it to improve their value and give more credible outputs to the company they are working in. You will take the input, which is the text of the research paper, and give the output in a summarized way, which would include:
    - what is the intent the paper wants to express
    - why this paper exists for the first time
    - what is the value addition that this paper can give you
    So first, make a plan to understand what are the items which are contained in the research paper and how you want to explain them to the end user. Then execute the plan to also inculcate the below items. 
    Make a section of basic knowledge that is required, where you will give subtopics and its understanding, which would help the machine, the user, or the fresh developer out of graduate to understand the research paper thoroughly and get a grip on the key words.
    Go on explaining the surface level of the paper to help him understand:
    - what is the abstract
    - what is the conclusion
    - what is the data used and results
    After completing this surface area level, enter into the research paper deeply by understanding:
    - what is the model architecture
    - check and review the inputs and outputs
    - what are the new techniques used
    - understand the loss functions
    - review the training parameters
    """

    user_prompt = f"""Please find the text of the research paper which starts after the colon within single quotes. Help me summarize it. '{text}' """
    messages = [
        {'role':'system','content':system_prompt},
        {'role':'user','content':user_prompt},
        ]
    
    return messages



In [ ]:
def streaming(Flag,pdf_url):
    ########### Authorization

    # OLLAMA_BASE_URL = "http://localhost:11434/v1"
    # ollama = OpenAI(api_key='ollama',base_url=OLLAMA_BASE_URL)
    ollama = OpenAI(api_key=api_key,base_url='https://openrouter.ai/api/v1')

    ############ Sending LLM Request
    # model='phi3'
    stream = ollama.chat.completions.create(
        model='openai/gpt-4o-mini-2024-07-18',
        messages=llm_msg(pdf_url),
        stream = Flag
    )
    
    # Response capture
    if Flag == True:
        response = ""
        display_handle = display(Markdown(""),display_id = True)
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            display_handle.update(Markdown(response))
    else:
        response = stream.choices[0].message.content
        display(Markdown(response))
    return response

In [ ]:
def ml_paper_summarizer(paper_name,stream = True):
    pdf_url = extract_url(paper_name)

    resp = streaming(stream,pdf_url)

    return resp

In [ ]:
# Inputs: Exact Research paper name from https://paperswithcode.co/ 
ml_paper_summarizer(paper_name = 'Adaptation of Agentic AI',stream = True)

### Comparing Model Outputs

In [ ]:
response = ollama.chat.completions.create(model='openai/gpt-4o-mini-2024-07-18',messages=llm_msg(pdf_url)) # response_format
print(response.usage)
# gpt4.0 mini response - 1min 
result = response.choices[0].message.content

In [ ]:
display(Markdown(result))

In [ ]:
# ollama response - 7.5 min
result = response.choices[0].message.content
print(result)

### Challenges to work on Local models vs Online hosted models:

**Inputs:**
1) Optimizing for sending higher context sizes than permited 

**Outputs:**
1) Improving output quality of local hosted LLM 
2) Optimizing inference time for higher context sizes
3) Reducing overall cost 

**Future Explorations:**
1. LLM to LLM calling
2. Implementation of RAG - optimizing for costs, time, quality, context size